# Project Overview and Methodology

This notebook is a meeting-friendly overview of the traffic signal control project. It explains how the simulator is built, what the baselines mean, what the current DQN setup is doing, and how to read the current results.

The current default experiment is a `1x1` single-intersection setting. The latest code also supports a `2x2` grid setting through `configs/grid_2x2.yaml`, but the checked-in `results/dqn_summary.json` is still a `1x1` run.

In [ ]:
import json
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
from IPython.display import Image, Markdown, display

PROJECT_ROOT = Path("..").resolve()
CONFIG_PATH = PROJECT_ROOT / "configs" / "default.yaml"
RESULTS_DIR = PROJECT_ROOT / "results"

baseline_path = RESULTS_DIR / "baseline_summary.json"
dqn_path = RESULTS_DIR / "dqn_summary.json"
training_plot_path = RESULTS_DIR / "plots" / "dqn" / "training_curves.png"
evaluation_plot_path = RESULTS_DIR / "plots" / "dqn" / "evaluation_comparison.png"

import sys
sys.path.insert(0, str(PROJECT_ROOT / "src"))

from traffic_rl.config import load_config


def load_json(path):
    if not path.exists():
        return None
    return json.loads(path.read_text(encoding="utf-8"))


def markdown_table(headers, rows):
    table = ["| " + " | ".join(headers) + " |"]
    table.append("| " + " | ".join(["---"] * len(headers)) + " |")
    for row in rows:
        table.append("| " + " | ".join(str(value) for value in row) + " |")
    return "\n".join(table)


config = load_config(CONFIG_PATH)
env_config = config["environment"]
train_config = config["training"]
eval_config = config["evaluation"]

baseline_summary = load_json(baseline_path)
dqn_summary = load_json(dqn_path)

display(Markdown(f"**Project root:** `{PROJECT_ROOT}`"))
display(Markdown(f"**Default config:** `{CONFIG_PATH.relative_to(PROJECT_ROOT)}`"))
display(Markdown(f"**Baseline summary found:** `{baseline_path.exists()}`"))
display(Markdown(f"**DQN summary found:** `{dqn_path.exists()}`"))


## 1. Research Goal

The project studies adaptive traffic signal control. At each time step, the controller decides whether to keep the current green phase or switch to the other phase. The goal is to reduce congestion and waiting time while avoiding unnecessary signal switching.

The main experimental question is:

> Can a DQN reinforcement learning controller learn a better long-horizon signal policy than simple rule-based traffic light controllers under different traffic demand regimes?

## 2. Environment Setup: What Is Being Simulated?

The default environment is a simplified queue-based simulator for one intersection.

It does **not** simulate continuous vehicle positions, acceleration, braking, driver reaction time, lane changing, or car-following physics. Instead, each approach of the intersection is represented as a queue.

The four queues are:

- `N`: vehicles arriving from the north approach
- `S`: vehicles arriving from the south approach
- `E`: vehicles arriving from the east approach
- `W`: vehicles arriving from the west approach

At every discrete time step, new vehicles are randomly added to these queues. Green-light approaches discharge vehicles up to a fixed service capacity. Red-light approaches continue waiting.

In [ ]:
env_rows = [
    ["network_type", env_config.get("network_type", "1x1"), "Default is a single intersection."],
    ["episode_length", env_config["episode_length"], "Number of simulation steps per episode."],
    ["step_seconds", env_config["step_seconds"], "Real-time duration represented by one step."],
    ["min_green_time", env_config["min_green_time"], "Minimum steps before a phase can switch."],
    ["yellow_time", env_config["yellow_time"], "Yellow-light transition duration in steps."],
    ["max_departures_per_step", env_config["max_departures_per_step"], "Maximum vehicles discharged per green approach per step."],
    ["reward_mode", env_config["reward_mode"], "Current reward penalty basis."],
    ["switch_penalty", env_config["switch_penalty"], "Extra penalty for a successful phase switch."],
    ["observation_variant", env_config["observation_variant"], "State representation used by DQN."],
]
display(Markdown("### Current Default Environment Parameters"))
display(Markdown(markdown_table(["parameter", "value", "meaning"], env_rows)))

episode_seconds = env_config["episode_length"] * env_config["step_seconds"]
display(Markdown(
    f"One episode is `{env_config['episode_length']}` steps, or `{episode_seconds}` seconds "
    f"({episode_seconds / 60:.1f} minutes)."
))


## 3. Signal Phases and Actions

The single-intersection simulator uses two signal phases:

- `phase 0`: north-south green, so `N` and `S` can discharge vehicles
- `phase 1`: east-west green, so `E` and `W` can discharge vehicles

The controller has two actions:

- `action 0 = keep`: keep the current phase
- `action 1 = switch`: request a switch to the other phase

A requested switch is only valid after `min_green_time` has passed and no yellow transition is active. If the switch is valid, the environment spends `yellow_time` steps in a transition. During yellow time, no vehicles are discharged.

In [ ]:
phase_rows = [
    ["phase 0", "N/S green", "N and S queues can discharge; E and W wait."],
    ["phase 1", "E/W green", "E and W queues can discharge; N and S wait."],
    ["action 0", "keep", "Continue the current phase."],
    ["action 1", "switch", "Request a phase change if switching is currently allowed."],
]
display(Markdown(markdown_table(["item", "meaning", "effect"], phase_rows)))

fig, ax = plt.subplots(figsize=(7, 3.2))
ax.axis("off")
ax.text(0.5, 0.88, "Single-Intersection Signal Phases", ha="center", fontsize=14, weight="bold")
ax.text(0.25, 0.55, "phase 0\nN/S green", ha="center", va="center", fontsize=12,
        bbox=dict(boxstyle="round,pad=0.4", fc="#d9f2e6", ec="#2a9d8f"))
ax.text(0.75, 0.55, "phase 1\nE/W green", ha="center", va="center", fontsize=12,
        bbox=dict(boxstyle="round,pad=0.4", fc="#fde2e2", ec="#e15759"))
ax.annotate("switch", xy=(0.63, 0.55), xytext=(0.37, 0.55), arrowprops=dict(arrowstyle="->"), ha="center")
ax.annotate("switch", xy=(0.37, 0.42), xytext=(0.63, 0.42), arrowprops=dict(arrowstyle="->"), ha="center")
plt.show()


## 4. How Vehicle Arrivals Are Simulated

Vehicle arrivals are sampled from a Poisson distribution. In this project, an arrival rate such as `0.8` means: on average, `0.8` vehicles arrive at that approach per simulation step.

Because one step is currently 3 seconds, a rate can be roughly converted to vehicles per minute:

`vehicles per minute = rate / step_seconds * 60`

For example, `rate = 0.8` means roughly `16` vehicles per minute on average for that approach. The actual number in each step is random: sometimes 0, sometimes 1, sometimes 2 or more.

In [ ]:
rates_to_explain = [0.4, 0.5, 0.7, 0.8, 1.2, 1.4, 2.0]
rate_rows = [
    [rate, f"{rate / env_config['step_seconds'] * 60:.1f}"]
    for rate in rates_to_explain
]
display(Markdown("### Interpreting Arrival Rates"))
display(Markdown(markdown_table(["arrival rate per step", "approx vehicles/minute"], rate_rows)))

rng = np.random.default_rng(7)
demo_rates = [0.5, 0.8, 1.4]
samples = {rate: rng.poisson(rate, size=80) for rate in demo_rates}

fig, axes = plt.subplots(1, 3, figsize=(13, 3.2), constrained_layout=True)
for ax, rate in zip(axes, demo_rates):
    values, counts = np.unique(samples[rate], return_counts=True)
    ax.bar(values, counts, color="#4e79a7")
    ax.set_title(f"Poisson arrivals, rate={rate}")
    ax.set_xlabel("vehicles arriving in one step")
    ax.set_ylabel("frequency in 80 steps")
    ax.set_xticks(range(0, max(values) + 1))
plt.show()


## 5. Service Capacity: `max_departures_per_step`

`max_departures_per_step` is a manually chosen simulator parameter. It is not calibrated from real traffic data in the current project.

It controls how many vehicles can leave each green approach during one step. With the default value `4`, each open approach can discharge up to four vehicles every three seconds.

Example: if the current phase is `phase 0`, then `N` and `S` are green. If `N` has 15 vehicles and `S` has 2 vehicles, the simulator discharges `min(15, 4) = 4` vehicles from `N` and `min(2, 4) = 2` vehicles from `S` in that step.

In [ ]:
example_queues = {"N": 15, "S": 2, "E": 8, "W": 10}
capacity = env_config["max_departures_per_step"]
phase0_departures = {direction: min(example_queues[direction], capacity) if direction in ["N", "S"] else 0 for direction in example_queues}
phase1_departures = {direction: min(example_queues[direction], capacity) if direction in ["E", "W"] else 0 for direction in example_queues}

display(Markdown("### Example Queue Service Under Each Phase"))
display(Markdown(markdown_table(
    ["direction", "starting queue", "departures if phase 0", "departures if phase 1"],
    [[d, example_queues[d], phase0_departures[d], phase1_departures[d]] for d in ["N", "S", "E", "W"]],
)))

x = np.arange(4)
directions = ["N", "S", "E", "W"]
fig, ax = plt.subplots(figsize=(8, 4))
ax.bar(x - 0.2, [phase0_departures[d] for d in directions], width=0.4, label="phase 0: N/S green")
ax.bar(x + 0.2, [phase1_departures[d] for d in directions], width=0.4, label="phase 1: E/W green")
ax.set_xticks(x)
ax.set_xticklabels(directions)
ax.set_ylabel("vehicles discharged in one step")
ax.set_title("Service Capacity Example")
ax.legend(frameon=False)
ax.grid(axis="y", alpha=0.2)
plt.show()


## 6. One Simulation Step

At a high level, one environment step follows this logic:

1. The controller observes the current queues and signal state.
2. The controller chooses `keep` or `switch`.
3. The environment checks whether switching is allowed.
4. Vehicles already waiting in queues age by one step.
5. New arrivals are sampled from the configured Poisson rates.
6. If the signal is in yellow, no vehicles are served.
7. Otherwise, the currently green approaches discharge vehicles up to `max_departures_per_step`.
8. New arrivals are appended to their queues.
9. The environment calculates reward and records metrics.
10. The next observation is returned.

This means waiting time is modeled at the queue level. Individual driver reaction time is not modeled directly.

## 7. Observation: What Does DQN See?

With `observation_variant: full`, the DQN receives a 13-dimensional state vector:

- queue lengths for `N`, `S`, `E`, `W`
- current signal phase
- current phase duration
- whether switching is currently allowed
- yellow time remaining
- normalized episode progress
- recent average arrivals for `N`, `S`, `E`, `W`

A smaller `minimal` observation variant also exists for ablation studies. It only includes queues, phase, and phase duration.

In [ ]:
observation_rows = [
    [0, "N queue length"],
    [1, "S queue length"],
    [2, "E queue length"],
    [3, "W queue length"],
    [4, "current phase"],
    [5, "phase duration"],
    [6, "switch allowed"],
    [7, "yellow remaining"],
    [8, "normalized episode step"],
    [9, "recent mean arrivals N"],
    [10, "recent mean arrivals S"],
    [11, "recent mean arrivals E"],
    [12, "recent mean arrivals W"],
]
display(Markdown(markdown_table(["index", "feature"], observation_rows)))


## 8. Reward Definition

The current default reward mode is `queue`.

The reward at each step is approximately:

`reward = - total_queue_length - switch_penalty if a valid switch was applied`

So the agent is encouraged to:

- keep queues short
- avoid unnecessary switching

There is also a `waiting` reward mode. In that mode, the reward penalizes accumulated waiting instead of directly penalizing total queue length.

In [ ]:
reward_examples = []
for total_queue in [5, 20, 40]:
    reward_no_switch = -total_queue
    reward_with_switch = -total_queue - env_config["switch_penalty"]
    reward_examples.append([total_queue, reward_no_switch, reward_with_switch])

display(Markdown(markdown_table(
    ["total queue", "reward without switch", "reward with valid switch"],
    reward_examples,
)))


## 9. Evaluation Regimes

`regime` means a traffic demand scenario. The default `1x1` config evaluates policies under five scenarios:

- `symmetric_low`: all directions have low traffic
- `symmetric_high`: all directions have high traffic
- `asymmetric`: east-west traffic is heavier than north-south traffic
- `nonstationary`: traffic rates change over time
- `burst_east_west`: east-west demand spikes during the middle of the episode

In [ ]:
def schedule_to_rows(name, schedule):
    rows = []
    previous = 0
    for segment in schedule:
        end = segment["until_step"]
        rates = segment["rates"]
        rows.append([
            name,
            f"{previous}-{end - 1}",
            rates.get("N", 0.0),
            rates.get("S", 0.0),
            rates.get("E", 0.0),
            rates.get("W", 0.0),
        ])
        previous = end
    return rows


schedule_rows = []
for regime, schedule in env_config["evaluation_regimes"].items():
    schedule_rows.extend(schedule_to_rows(regime, schedule))

display(Markdown(markdown_table(["regime", "step range", "N", "S", "E", "W"], schedule_rows)))

fig, ax = plt.subplots(figsize=(10, 4.8))
for regime, schedule in env_config["evaluation_regimes"].items():
    if len(schedule) == 1:
        rates = schedule[0]["rates"]
        ax.plot([regime] * 4, [rates[d] for d in ["N", "S", "E", "W"]], "o", label=regime)
ax.set_ylabel("arrival rate per step")
ax.set_title("Stationary Evaluation Regime Rates")
ax.grid(axis="y", alpha=0.2)
plt.xticks(rotation=20, ha="right")
plt.show()


## 10. Baseline Controllers

In this project, `baseline` means a non-learning rule-based traffic signal controller. These are not un-tuned DQN runs. They are heuristic policies used as comparison points for DQN.

The default baselines are:

- `fixed_cycle`: switch after a fixed green duration
- `queue_threshold`: switch if the opposite direction is much more congested
- `max_pressure`: switch toward the direction with larger queue pressure

The current project uses one RL method for this task: DQN. The performance comparison is DQN versus the rule-based baselines.

In [ ]:
baseline_rows = [
    ["fixed_cycle", "cycle_length=10", "Switch after a fixed number of green steps if switching is allowed."],
    ["queue_threshold", "threshold=5.0, min_green=3", "Switch when the opposite axis has more than 5 extra queued vehicles."],
    ["max_pressure", "min_green=2", "Serve the axis with the larger total queue once switching is allowed."],
]
display(Markdown(markdown_table(["baseline", "main parameters", "logic"], baseline_rows)))


## 11. RL Method: DQN

The current RL method is Deep Q-Network (DQN). The project does not currently compare multiple RL algorithms such as PPO, A2C, or SARSA. The main comparison is:

`DQN` vs. `fixed_cycle` vs. `queue_threshold` vs. `max_pressure`

DQN learns a Q-value for each action from the current observation. It uses replay buffer training, a target network, epsilon-greedy exploration, and action masking to avoid invalid signal switches.

In [ ]:
training_rows = [
    ["episodes", train_config["episodes"]],
    ["gamma", train_config["gamma"]],
    ["learning_rate", train_config["learning_rate"]],
    ["batch_size", train_config["batch_size"]],
    ["buffer_size", train_config["buffer_size"]],
    ["hidden_dims", train_config["hidden_dims"]],
    ["start_epsilon", train_config["start_epsilon"]],
    ["end_epsilon", train_config["end_epsilon"]],
    ["epsilon_decay_steps", train_config["epsilon_decay_steps"]],
    ["warmup_steps", train_config["warmup_steps"]],
    ["target_sync_steps", train_config["target_sync_steps"]],
    ["seed", train_config["seed"]],
]
display(Markdown("### Current Default DQN Parameters"))
display(Markdown(markdown_table(["parameter", "value"], training_rows)))

display(Markdown(
    "The checked-in `results/dqn_summary.json` appears to be a default full training run, "
    "not a confirmed best-tuned run. The config has a tuning section, but no checked-in "
    "`results/tuning/tuning_summary.json` is currently present."
))


## 12. Current Result Files

The notebook loads result files if they already exist. If they are missing, run the corresponding scripts from the project root:

```bash
python3 scripts/run_baselines.py --config configs/default.yaml
python3 scripts/train_dqn.py --config configs/default.yaml
python3 scripts/plot_results.py --summary results/dqn_summary.json --output-dir results/plots/dqn
```

In [ ]:
if baseline_summary is None:
    display(Markdown("Baseline results are missing. Run `python3 scripts/run_baselines.py --config configs/default.yaml`."))
else:
    rows = []
    for regime, policy_results in baseline_summary.items():
        for policy, metrics in policy_results.items():
            rows.append([
                regime,
                policy,
                f"{metrics['average_queue_length']:.2f}",
                f"{metrics['average_wait_time_seconds']:.2f}",
                f"{metrics['throughput_per_step']:.2f}",
                f"{metrics['switch_count']:.2f}",
                f"{metrics.get('invalid_switch_count', 0.0):.2f}",
            ])
    display(Markdown("### Baseline Results"))
    display(Markdown(markdown_table(
        ["regime", "policy", "avg_queue", "avg_wait_s", "throughput", "switches", "invalid"],
        rows,
    )))


In [ ]:
if dqn_summary is None:
    display(Markdown("DQN results are missing. Run `python3 scripts/train_dqn.py --config configs/default.yaml`."))
else:
    training_history = dqn_summary.get("training_history", [])
    final = training_history[-1] if training_history else {}
    metadata = dqn_summary.get("metadata", {})
    overview_rows = [
        ["episodes", len(training_history)],
        ["checkpoint", dqn_summary.get("checkpoint", "")],
        ["network_type", metadata.get("network_type", "1x1")],
        ["train_schedule", metadata.get("train_schedule_name", "n/a")],
        ["reward_mode", metadata.get("reward_mode", "n/a")],
        ["observation_variant", metadata.get("observation_variant", "n/a")],
        ["final_reward", f"{final.get('total_reward', 0.0):.2f}"],
        ["final_avg_queue", f"{final.get('average_queue_length', 0.0):.2f}"],
        ["final_avg_wait_s", f"{final.get('average_wait_time_seconds', 0.0):.2f}"],
        ["final_invalid_switches", f"{final.get('invalid_switch_count', 0.0):.2f}"],
    ]
    display(Markdown("### DQN Training Overview"))
    display(Markdown(markdown_table(["field", "value"], overview_rows)))


In [ ]:
if training_plot_path.exists():
    display(Markdown("### Existing DQN Training Curves"))
    display(Image(filename=str(training_plot_path)))
else:
    display(Markdown("Training curve image is missing. Run `python3 scripts/train_dqn.py --config configs/default.yaml`."))

if evaluation_plot_path.exists():
    display(Markdown("### Existing Evaluation Comparison"))
    display(Image(filename=str(evaluation_plot_path)))
else:
    display(Markdown("Evaluation comparison image is missing. Run `python3 scripts/train_dqn.py --config configs/default.yaml`."))


In [ ]:
if dqn_summary is not None:
    eval_results = dqn_summary.get("evaluation_results", {})
    regimes = list(eval_results.keys())
    policies = list(next(iter(eval_results.values())).keys()) if eval_results else []
    metrics = [
        ("average_queue_length", "Average Queue Length"),
        ("average_wait_time_seconds", "Average Wait Time (s)"),
        ("throughput_per_step", "Throughput per Step"),
    ]

    if regimes and policies:
        fig, axes = plt.subplots(1, 3, figsize=(16, 5), constrained_layout=True)
        x = np.arange(len(regimes))
        width = 0.18
        for axis, (metric_key, title) in zip(axes, metrics):
            for idx, policy in enumerate(policies):
                values = [eval_results[regime][policy][metric_key] for regime in regimes]
                axis.bar(x + (idx - (len(policies) - 1) / 2) * width, values, width, label=policy)
            axis.set_title(title)
            axis.set_xticks(x)
            axis.set_xticklabels(regimes, rotation=25, ha="right")
            axis.grid(axis="y", alpha=0.2)
        axes[0].legend(frameon=False)
        plt.show()


## 13. Suggested Next Steps

1. Clean duplicate local files with ` 2` suffixes after confirming they are not needed.
2. Re-run baseline and DQN experiments from the latest code version.
3. Run `scripts/tune_dqn.py` to produce a documented tuning summary and best config.
4. Re-run the final `1x1` DQN using the selected best config.
5. Use `configs/grid_2x2.yaml` and `--profile 2x2` for the grid extension once the `1x1` methodology and results are stable.
6. Export selected tables and figures from this notebook into slides.